# Module 1 GenAI on Databricks: The Full Stack
### Setup, Unity Catalog, and your first traced LLM call — fully hands-on

This notebook is meant to be **run cell by cell, not "Run All."** Along the way you'll be sent
into the Databricks UI itself — Catalog Explorer, Compute, Serving, Experiments — so the
platform tour isn't just code, it's clicking around too.


## Step 0 — Open a second browser tab now
Keep the Databricks workspace open in a second tab for the rest of this module. You'll be told
exactly where to click and what to look for as you go — having it open side-by-side is much
faster than switching notebooks back and forth.

## 1. Configuration widgets
Run the cell below once. It creates **interactive widgets at the top of this notebook** —
Databricks renders these as an actual form (text boxes and a dropdown) above the cell. Change any
value there any time and re-run the notebook from this point on to use it.

In [0]:
dbutils.widgets.text("catalog", "genai_course", "Catalog name")
dbutils.widgets.text("schema", "rag_demo", "Schema name")
dbutils.widgets.text("volume", "source_docs", "Volume name (for raw files)")
dbutils.widgets.dropdown(
    "llm_endpoint",
    "system.ai.claude-sonnet-4-5",
    [
        "system.ai.claude-sonnet-4-5",
        "system.ai.claude-opus-4-6",
        "system.ai.claude-haiku-4-5",
        "system.ai.meta-llama-3-3-70b-instruct",
    ],
    "Chat model service (Unity Gateway, system.ai schema)",
)

print("✓ Widgets created — look above this cell for the form, then continue below.")

In [0]:
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME = dbutils.widgets.get("volume")
LLM_ENDPOINT = dbutils.widgets.get("llm_endpoint")

print(f"Three-level namespace for this course : {CATALOG}.{SCHEMA}.<object>")
print(f"Raw files will live in                : /Volumes/{CATALOG}/{SCHEMA}/{VOLUME}")
print(f"Chat model endpoint                   : {LLM_ENDPOINT}")

 **Model names above are examples — confirm your own.** Databricks now serves Foundation Models
through **Unity Gateway**: ready-to-query "model services" that live in Unity Catalog under the
`system.ai` schema, named like `system.ai.claude-sonnet-4-5`. You do **not** create a serving
endpoint to use them — that's why a fresh workspace's **Serving** page shows "No endpoints found."
(That page is now mainly for endpoints *you* deploy yourself — like the agent you'll build in
Module 3.)

## Confirm your model names
1. Click **Catalog** in the left sidebar.
2. Expand the **system** catalog → **ai** schema.
3. Every row there is a model you can query right now, no setup — the widget dropdown above lists
   a few common ones, but yours may differ by workspace/region. Pick a real one from this list and
   re-run the cell above if it doesn't match.
4. Alternatively: **Playground** in the sidebar lets you try any of them in a chat UI first.

## 2. Confirm your compute and permissions
A quick sanity check before we create anything.

In [0]:
current_user = spark.sql("SELECT current_user() AS user").collect()[0]["user"]
print(f"Running as: {current_user}")

checks = {}
try:
    spark.sql(f"SHOW CATALOGS LIKE '{CATALOG}'")
    checks["can_list_catalogs"] = True
except Exception as e:
    checks["can_list_catalogs"] = False
    print(f"✗ {e}")

print("✅ Can list catalogs" if checks["can_list_catalogs"] else "❌ Fix catalog permissions before continuing")

## 3. Create the Unity Catalog objects this course uses
This is the one cell every later module depends on. Run it once, now.

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}
    COMMENT 'Raw source documents for the GenAI on Databricks course'
""")

print(f"✓ {CATALOG} catalog ready")
print(f"✓ {CATALOG}.{SCHEMA} schema ready")
print(f"✓ {CATALOG}.{SCHEMA}.{VOLUME} volume ready")

## See what you just created, in the UI
1. Click **Catalog** in the left sidebar.
2. Expand your catalog (`genai_course` by default) → your schema (`rag_demo`) → **Volumes**.
3. You should see `source_docs` sitting there, empty.
4. Click on it — this is the same Catalog Explorer screen you'll use in every later module to
   check on tables, functions, and models as you create them.

## ✅ Checkpoint 1 — catalog objects exist
Run this — it queries Unity Catalog directly rather than trusting the cell above didn't silently fail.

In [0]:
result = spark.sql(f"""
    SELECT catalog_name, schema_name
    FROM system.information_schema.schemata
    WHERE catalog_name = '{CATALOG}' AND schema_name = '{SCHEMA}'
""").collect()

if result:
    print(f"✅ PASS — {CATALOG}.{SCHEMA} is visible in Unity Catalog")
else:
    print(f"❌ FAIL — {CATALOG}.{SCHEMA} not found. Re-run the cell in Step 3.")

## Try it yourself: upload a real file
1. Back in **Catalog Explorer**, open `{catalog}.{schema}.source_docs` (from the UI step above).
2. Click **Upload to this volume** and drag in any PDF you have handy — a policy doc, a manual,
   anything with a few pages of text works for this course.
3. Come back here and run the next cell to confirm the notebook can see it too.

In [0]:
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
files = dbutils.fs.ls(VOLUME_PATH)

if files:
    print(f"✅ Found {len(files)} file(s) in {VOLUME_PATH}:")
    for f in files:
        print(f"   • {f.name}  ({f.size:,} bytes)")
else:
    print(f"⚠️  {VOLUME_PATH} is empty — upload a PDF in the UI step above, then re-run this cell.")

## 4. Turn on MLflow tracing
Two lines. Every LLM call for the rest of this course gets automatically captured as a trace —
inputs, outputs, latency, token counts — with no extra code at the call site.

In [0]:
import mlflow

mlflow.openai.autolog()
mlflow.set_registry_uri("databricks-uc")

print("✓ Tracing enabled — every call below will show up under Experiments.")

## 4b. A reusable client for querying Unity Gateway model services
This is the one helper every module in this course reuses to talk to `system.ai.*` models. It
builds a plain OpenAI-compatible client pointed at your workspace's Unity Gateway path, using the
notebook's own auth context — no token to paste in.

In [0]:
from openai import OpenAI

def get_chat_client():
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    workspace_url = ctx.apiUrl().get()
    token = ctx.apiToken().get()
    return OpenAI(api_key=token, base_url=f"{workspace_url}/ai-gateway/mlflow/v1")

openai_client = get_chat_client()
print("✓ Client ready — points at Unity Gateway, not a classic Serving endpoint.")

In [0]:
import mlflow
mlflow.openai.autolog()

## Try it yourself: ask your own question
Run the widget cell below once, then look **above this notebook** for a new text box called
`user_question`. Type any question into it, then re-run the code cell underneath as many times as
you like — no need to re-run anything above.

In [0]:
dbutils.widgets.text("user_question", "In one sentence, what is delta table?", "Ask the model anything")

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

resp = w.api_client.do(
    "GET",
    "/api/2.1/unity-catalog/model-services",
    query={"parent": "schemas/system.ai"},
)

services = resp.get("model_services", [])
print(f"Found {len(services)} model service(s)\n")

# Dump the raw first item so we can see the real field names
import json
print(json.dumps(services[0], indent=2))

In [0]:
for m in services:
    print(m["name"].removeprefix("model-services/"))

In [0]:
LLM_ENDPOINT = "system.ai.gpt-oss-20b"  # smallest, cheapest, most likely to be open

question = dbutils.widgets.get("user_question")

response = openai_client.chat.completions.create(
    model=LLM_ENDPOINT,
    messages=[
        {"role": "system", "content": "You are a concise assistant for a Databricks GenAI course."},
        {"role": "user", "content": question},
    ],
    max_tokens=200,
)

print(f"Q: {question}\n")
print(f"A: {response.choices[0].message.content}")

👆 **Change the `user_question` widget above and re-run just that last cell a few times.** Try:
- *"What's the difference between a catalog and a schema?"*
- *"Explain Delta Lake in two sentences."*
- Something totally unrelated, just to see the model respond

Each call is a separate trace — you're about to go look at all of them at once.

## See the traces you just created
1. Click **Experiments** in the left sidebar (under **AI/ML**).
2. Open the experiment matching this notebook's path.
3. Click the **Traces** tab.
4. You should see one row per question you asked. Click into any one — you'll see the exact
   prompt sent, the exact response, latency, and token counts.

## ✅ Checkpoint 2 — pull the same traces back into the notebook
You don't have to leave the notebook to see this — `mlflow.search_traces` reads the same data the UI shows.

In [0]:
traces = mlflow.search_traces(max_results=10)

if len(traces):
    print(f"✅ PASS — found {len(traces)} trace(s) for this session")
    display(traces[["request_time", "state", "execution_duration"]])
else:
    print("❌ FAIL — no traces found. Did the autolog() cell run before your question cell?")

## ✅ Final report card
Run this last cell — it re-checks everything in this notebook in one shot.

In [0]:
report = []

# Catalog / schema
exists = spark.sql(f"""
    SELECT 1 FROM system.information_schema.schemata
    WHERE catalog_name = '{CATALOG}' AND schema_name = '{SCHEMA}'
""").collect()
report.append(("Catalog & schema created", bool(exists)))

# Volume has a file
try:
    has_files = len(dbutils.fs.ls(f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}")) > 0
except Exception:
    has_files = False
report.append(("At least one file uploaded to the volume", has_files))

# Traces exist
report.append(("At least one MLflow trace captured", len(mlflow.search_traces(max_results=1)) > 0))

print("MODULE 1 — REPORT CARD")
print("=" * 40)
all_pass = True
for label, passed in report:
    icon = "✅" if passed else "❌"
    print(f"{icon}  {label}")
    all_pass = all_pass and passed

print("=" * 40)
print("🎉 Ready for Module 2!" if all_pass else "⚠️  Fix the ❌ items above before Module 2.")

---
### What's next
**Module 2** builds directly on `{CATALOG}.{SCHEMA}.{VOLUME}` and the PDF you just uploaded —
parsing it, chunking it, embedding it, and making it searchable.